# Recording Na⁺ and K⁺ ionic currents during an AP

Copyright (c) 2025 Open Brain Institute

Authors: Ilkan Kilic

last modified: 08.2025

## Summary
In this example, we simulate a single neuron (me-model) and record its membrane potential (v) together with sodium (Na⁺, ina) and potassium (K⁺, ik) ionic currents. By observing these ion-level currents during an evoked action potential, we can explore how sodium influx and potassium efflux shape the waveform of the spike.

In [ ]:
import numpy
import pandas
import json

from ipywidgets import widgets, interact
from matplotlib import pyplot as plt

from obi_auth import get_token
from entitysdk.client import Client
from entitysdk.models import SingleNeuronSimulation
from obi_notebook import get_projects
from obi_notebook import get_entities

token = get_token(environment="production", auth_mode="daf")
project_context = get_projects.get_projects(token)

## Load the simulation result

In [ ]:
client = Client(
    project_context=project_context,
    environment="staging",
    token_manager=token,
)

simulation_ids = [
    # put here the id you want to use as a string
    # you can add as many simulation IDs as you want

]
# Alternative: Select from a table of entities
if not simulation_ids:
    simulation_ids = get_entities.get_entities("single-neuron-simulation", token, simulation_ids,
                                            project_context=project_context,
                                            multi_select=True,
                                            page_size=100)

In [ ]:
simulation_paths = []
for sim_id in simulation_ids:
    simulation = client.get_entity(
        entity_type=SingleNeuronSimulation,
        entity_id=sim_id,
    )
    asset = client.download_assets(
        simulation,
        output_path="./",  # here you can put a repo for downloading the simulation
    ).one()
    simulation_paths.append(asset.path)

Retrieve the recorded data

In [ ]:
data_keys = ["x", "y"]

def entries_to_df(entries):
    data = []
    for entry in entries:
        df = pandas.DataFrame(dict([(k, entry.pop(k)) for k in data_keys]))
        for k, v in entry.items():
            df[k] = v
        data.append(df)
    ret = pandas.concat(data, axis=0)
    cols = [_c for _c in ret.columns if _c not in data_keys]
    ret = ret.fillna("_NONE").set_index(cols)
    return ret

def read_list_of_entries(lst):
    data = pandas.concat([entries_to_df(_v) for _v in lst.values()], axis=0)
    return data

def read_sim_config_data(fid):
    cfg = json.load(fid)
    data_out = read_list_of_entries(cfg["simulation"])
    stim_out = entries_to_df(cfg["stimulus"])
    return data_out, stim_out

all_data = []; all_stim = []
for sim_path in simulation_paths:
    with open(sim_path) as fid:
        data, stim = read_sim_config_data(fid)
        all_data.append(data); all_stim.append(stim)

In [ ]:
all_data

In [ ]:
todo check this notebook : https://github.com/openbraininstitute/obi_platform_analysis_notebooks/blob/main/Cellular/analyse_single_cell_sim/analysis_notebook.ipynb

Retrieve and plot the results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

# Membrane potential
plt.subplot(3, 1, 1)
plt.plot(t, v_trace, color="black")
plt.ylabel("V (mV)")
plt.title("Action potential with Na⁺ / K⁺ current densities")

# Sodium current density
plt.subplot(3, 1, 2)
plt.plot(t, ina_density, color="blue")
plt.ylabel("iNa (mA/cm²)")

# Potassium current density
plt.subplot(3, 1, 3)
plt.plot(t, ik_density, color="red", alpha=0.6)
plt.ylabel("iK (mA/cm²)")
plt.xlabel("Time (ms)")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Select a small time window around one AP (adjust as needed) ---
t_zoom_start, t_zoom_end = 125, 135  # ms
mask = (t >= t_zoom_start) & (t <= t_zoom_end)

# --- Plot ---
fig, ax1 = plt.subplots(figsize=(8, 6))

# Membrane potential (left y-axis)
ax1.plot(t[mask], v_trace[mask], label="Membrane V", color="black")
ax1.set_ylabel("V (mV)")
ax1.set_xlabel("Time (ms)")

# Na⁺ and K⁺ current densities (right y-axis)
ax2 = ax1.twinx()
ax2.plot(t[mask], ina_density[mask], label="iNa (mA/cm²)", color="blue")
ax2.plot(t[mask], ik_density[mask], label="iK (mA/cm²)", color="red", alpha=0.6)
ax2.set_ylabel("Current density (mA/cm²)")

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.title("AP with Na⁺ and K⁺ current densities (zoomed)")
plt.tight_layout()
plt.show()

The plot shows a zoomed view of one action potential. The membrane potential (black) rises sharply, accompanied by a brief inward Na⁺ current density (blue, small negative deflection) and a delayed outward K⁺ current density (red, positive deflection in mA/cm²) that repolarizes the cell.